In [1]:
import logging
from abc import ABC, abstractmethod
from typing import Dict, Any, Optional, List

# 配置全局日志
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger("ToolSystem")


# ==============================================================================
# 1. Tool 基类与具体工具实现
# ==============================================================================

class Tool(ABC):
    """Tool 抽象基类"""

    def __init__(self, name: str, description: str, required_params: Optional[List[str]] = None):
        self.name = name
        self.description = description
        self.required_params = required_params or []

    @abstractmethod
    def execute(self, **kwargs) -> Any:
        """子类必须实现的具体执行逻辑"""
        pass


class EmployeeTool(Tool):
    """员工信息查询工具"""

    def __init__(self):
        super().__init__(
            name="get_employee_info",
            description="根据员工 ID 查询员工基本信息",
            required_params=["employee_id"]
        )

    def execute(self, employee_id: str, **kwargs) -> dict:
        # 模拟业务逻辑
        if employee_id == "E1001":
            return {"employee_id": "E1001", "name": "Kevin", "dept": "Corp.IT", "role": "Developer"}
        raise ValueError(f"未找到 ID 为 '{employee_id}' 的员工数据")


class WeatherTool(Tool):
    """天气查询工具"""

    def __init__(self):
        super().__init__(
            name="get_weather",
            description="查询指定城市的天气",
            required_params=["city"]
        )

    def execute(self, city: str, **kwargs) -> dict:
        # 模拟业务逻辑
        mock_weather = {
            "上海": {"temperature": "28°C", "condition": "多云"},
            "北京": {"temperature": "25°C", "condition": "晴"}
        }
        if city in mock_weather:
            return {"city": city, **mock_weather[city]}
        return {"city": city, "temperature": "22°C", "condition": "未知"}


class OrderTool(Tool):
    """订单查询工具"""

    def __init__(self):
        super().__init__(
            name="get_order_details",
            description="根据订单号查询订单状态",
            required_params=["order_id"]
        )

    def execute(self, order_id: str, **kwargs) -> dict:
        # 模拟业务逻辑
        if order_id.startswith("ORD"):
            return {"order_id": order_id, "status": "COMPLETED", "amount": "$20.00"}
        raise ValueError(f"无效的订单编号格式: '{order_id}'")


# ==============================================================================
# 2. ToolRegistry (注册表)
# ==============================================================================

class ToolRegistry:
    """负责保存、注册、查找 Tool"""

    def __init__(self):
        self._tools: Dict[str, Tool] = {}

    def register(self, tool: Tool) -> None:
        """注册工具"""
        if tool.name in self._tools:
            logger.warning(f"[Registry] 工具 '{tool.name}' 已存在，将被覆盖。")
        self._tools[tool.name] = tool
        logger.info(f"[Registry] 成功注册工具: [{tool.name}]")

    def get_tool(self, name: str) -> Optional[Tool]:
        """查找工具"""
        return self._tools.get(name)

    def list_tools(self) -> Dict[str, str]:
        """获取所有已注册工具列表及描述"""
        return {name: tool.description for name, tool in self._tools.items()}


# ==============================================================================
# 3. ToolManager (管理者)
# ==============================================================================

class ToolManager:
    """负责校验、执行、异常处理、日志输出"""

    def __init__(self, registry: ToolRegistry):
        self.registry = registry

    def _validate_args(self, tool: Tool, kwargs: dict) -> None:
        """校验必要参数"""
        missing = [param for param in tool.required_params if param not in kwargs or kwargs[param] is None]
        if missing:
            raise ValueError(f"工具 [{tool.name}] 缺少必要参数: {', '.join(missing)}")

    def invoke(self, tool_name: str, **kwargs) -> dict:
        """
        统一调用入口，完成：查找 -> 校验 -> 执行 -> 捕获异常 -> 记录日志
        """
        logger.info(f"[Manager] ==> 请求调用工具: [{tool_name}] | 参数: {kwargs}")

        # 1. 查找工具
        tool = self.registry.get_tool(tool_name)
        if not tool:
            err_msg = f"未找到名为 '{tool_name}' 的工具"
            logger.error(f"[Manager] <== 执行失败: {err_msg}")
            return {"success": False, "error": err_msg, "data": None}

        try:
            # 2. 参数校验
            self._validate_args(tool, kwargs)

            # 3. 执行工具
            result = tool.execute(**kwargs)

            logger.info(f"[Manager] <== 工具 [{tool_name}] 执行成功 | 结果: {result}")
            return {"success": True, "error": None, "data": result}

        except Exception as e:
            # 4. 统一捕获异常并记录日志
            logger.error(f"[Manager] <== 工具 [{tool_name}] 执行过程中抛出异常: {str(e)}")
            return {"success": False, "error": str(e), "data": None}


# ==============================================================================
# 4. 演示运行 (Demo)
# ==============================================================================

if __name__ == "__main__":
    # 实例化注册表与管理者
    registry = ToolRegistry()
    manager = ToolManager(registry)

    # 1. 注册所有工具
    registry.register(EmployeeTool())
    registry.register(WeatherTool())
    registry.register(OrderTool())

    print("\n" + "=" * 50)
    print("已注册工具清单:", registry.list_tools())
    print("=" * 50 + "\n")

    # 2. 正常调用示例
    manager.invoke("get_employee_info", employee_id="E1001")
    manager.invoke("get_weather", city="上海")
    manager.invoke("get_order_details", order_id="ORD-20260714")

    print("\n--- 以下演示异常捕获场景 ---\n")

    # 3. 缺少必要参数
    manager.invoke("get_weather")

    # 4. 业务内部抛出异常
    manager.invoke("get_employee_info", employee_id="E9999")

    # 5. 调用未注册工具
    manager.invoke("get_unknown_data")

[2026-07-24 16:06:33] [INFO] [Registry] 成功注册工具: [get_employee_info]
[2026-07-24 16:06:33] [INFO] [Registry] 成功注册工具: [get_weather]
[2026-07-24 16:06:33] [INFO] [Registry] 成功注册工具: [get_order_details]
[2026-07-24 16:06:33] [INFO] [Manager] ==> 请求调用工具: [get_employee_info] | 参数: {'employee_id': 'E1001'}
[2026-07-24 16:06:33] [INFO] [Manager] <== 工具 [get_employee_info] 执行成功 | 结果: {'employee_id': 'E1001', 'name': 'Kevin', 'dept': 'Corp.IT', 'role': 'Developer'}
[2026-07-24 16:06:33] [INFO] [Manager] ==> 请求调用工具: [get_weather] | 参数: {'city': '上海'}
[2026-07-24 16:06:33] [INFO] [Manager] <== 工具 [get_weather] 执行成功 | 结果: {'city': '上海', 'temperature': '28°C', 'condition': '多云'}
[2026-07-24 16:06:33] [INFO] [Manager] ==> 请求调用工具: [get_order_details] | 参数: {'order_id': 'ORD-20260714'}
[2026-07-24 16:06:33] [INFO] [Manager] <== 工具 [get_order_details] 执行成功 | 结果: {'order_id': 'ORD-20260714', 'status': 'COMPLETED', 'amount': '$20.00'}
[2026-07-24 16:06:33] [INFO] [Manager] ==> 请求调用工具: [get_weather] | 参数: {


已注册工具清单: {'get_employee_info': '根据员工 ID 查询员工基本信息', 'get_weather': '查询指定城市的天气', 'get_order_details': '根据订单号查询订单状态'}


--- 以下演示异常捕获场景 ---

